# Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Sequential Data Preparation

In [3]:
# --- 1. Load & Split ---
interactions_path = Path.cwd().parent/"data"/"interactions_train.csv"
df = pd.read_csv(interactions_path)
df = df.rename(columns={"u": "user_id", "i": "item_id", "t": "timestamp"})

df = df.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)

df['rank'] = df.groupby('user_id')['timestamp'].rank(method='first', ascending=True)
df['total'] = df.groupby('user_id')['timestamp'].transform('count')

train_df = df[df['rank'] <= df['total'] * 0.8].copy()
val_df = df[df['rank'] > df['total'] * 0.8].copy()

# --- 2. Encoding ---
user2idx = {u: i for i, u in enumerate(df['user_id'].unique())}
idx2user = {i: u for i, u in enumerate(df['user_id'].unique())}
item2idx = {i: idx for idx, i in enumerate(df['item_id'].unique())}
idx2item = {idx: i for idx, i in enumerate(df['item_id'].unique())}

num_users, num_items = len(user2idx), len(item2idx)

train_df['user_idx'] = train_df['user_id'].map(user2idx)
# SHIFT ITEMS BY +1 FOR PADDING (0)
train_df['item_idx'] = train_df['item_id'].map(item2idx) + 1 
val_df['user_idx'] = val_df['user_id'].map(user2idx)
val_df['item_idx'] = val_df['item_id'].map(item2idx) + 1

# --- 3. Create Sequences ---
max_seq_len = 15

# Group into ordered lists
train_history = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
val_history = val_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
full_history = df.copy()
full_history['item_idx'] = full_history['item_id'].map(item2idx) + 1
full_user_history = full_history.groupby('user_id')['item_idx'].apply(list).to_dict()

def create_sasrec_data(history_dict, max_len):
    users, seqs, targets = [], [], []
    for user, history in history_dict.items():
        if len(history) < 2:
            continue # Need at least 1 input and 1 target
        
        seq = history[:-1]
        target = history[-1]
        
        # Pad or Truncate
        if len(seq) > max_len:
            seq = seq[-max_len:]
        else:
            seq = [0] * (max_len - len(seq)) + seq # Pad with 0s at the front
            
        users.append(user)
        seqs.append(seq)
        targets.append(target)
        
    return torch.tensor(users), torch.tensor(seqs), torch.tensor(targets)

u_t, seq_t, target_t = create_sasrec_data(train_history, max_seq_len)

class SASRecDataset(Dataset):
    def __init__(self, u, seq, target):
        self.u, self.seq, self.target = u, seq, target
    def __len__(self): return len(self.u)
    def __getitem__(self, idx): return self.u[idx], self.seq[idx], self.target[idx]

train_loader = DataLoader(SASRecDataset(u_t, seq_t, target_t), batch_size=512, shuffle=True)

# Micro-Transformer Model
This is a highly constrained transformer designed specifically to extract signal from short sequences. (A massive transformer will instantly overfit an average of 11 interactions)

In [4]:
class SASRecSmall(nn.Module):
    def __init__(self, num_items, max_seq_len, embed_dim=32, dropout=0.3):
        super(SASRecSmall, self).__init__()
        
        # item_emb size is num_items + 1 to account for the 0 padding index
        self.item_emb = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.emb_dropout = nn.Dropout(dropout)
        
        # Tiny transformer: 1 layer, 1 head
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=1, 
            dim_feedforward=embed_dim * 2, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, seqs):
        batch_size, seq_len = seqs.size()
        
        seq_embs = self.item_emb(seqs)
        positions = torch.arange(seq_len, device=seqs.device).unsqueeze(0).expand(batch_size, seq_len)
        pos_embs = self.pos_emb(positions)
        
        x = self.emb_dropout(seq_embs + pos_embs)
        
        # Causal mask so it can't look at future books in the sequence
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(seqs.device)
        padding_mask = (seqs == 0)
        
        out = self.transformer(x, mask=mask, src_key_padding_mask=padding_mask)
        
        # We only care about the very last sequence output to predict the NEXT item
        final_out = self.layer_norm(out[:, -1, :])
        
        # Dot product with all items to get scores
        logits = torch.matmul(final_out, self.item_emb.weight.T)
        return logits

# Training and Evaluation Loop

In [5]:
def eval_sasrec(model, val_history, train_history, max_len, device):
    model.eval()
    precisions = []
    with torch.no_grad():
        for user_idx, true_items in val_history.items():
            # Use their full training history as the input sequence
            seq = train_history.get(user_idx, [])
            if not seq: continue
                
            if len(seq) > max_len: seq = seq[-max_len:]
            else: seq = [0] * (max_len - len(seq)) + seq
                
            seq_tensor = torch.tensor([seq], device=device)
            logits = model(seq_tensor).squeeze()
            
            # Mask out padding (0) and items they already read in training
            logits[0] = -float('inf') 
            past_items = train_history.get(user_idx, [])
            if past_items:
                logits[past_items] = -float('inf')
                
            top_10 = torch.topk(logits, k=10).indices.cpu().numpy()
            hits = len(set(top_10).intersection(set(true_items)))
            precisions.append(hits / 10.0)
    return np.mean(precisions)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SASRecSmall(num_items, max_seq_len, embed_dim=32).to(device)

# Standard Cross Entropy is perfect here
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

epochs = 15
best_p10 = 0

print(f"Training SASRec on {device}")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for _, seqs, targets in train_loader:
        seqs, targets = seqs.to(device), targets.to(device)
        optimizer.zero_grad()
        
        logits = model(seqs)
        loss = criterion(logits, targets)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    val_p10 = eval_sasrec(model, val_history, train_history, max_seq_len, device)
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss/len(train_loader):.4f} | Val P@10: {val_p10:.5f}")
    
    if val_p10 > best_p10:
        best_p10 = val_p10
        torch.save(model.state_dict(), 'best_sasrec.pth')

C:\Users\Julien\AppData\Local\Temp\ipykernel_13704\2015910104.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)


Training SASRec on cuda


e:\anaconda3\envs\RecoSys\Lib\site-packages\torch\nn\modules\transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


Epoch 01 | Loss: 20.3467 | Val P@10: 0.00009
Epoch 02 | Loss: 18.6042 | Val P@10: 0.00003
Epoch 03 | Loss: 16.8620 | Val P@10: 0.00004
Epoch 04 | Loss: 15.2902 | Val P@10: 0.00006
Epoch 05 | Loss: 14.0718 | Val P@10: 0.00013
Epoch 06 | Loss: 13.0344 | Val P@10: 0.00015
Epoch 07 | Loss: 12.2632 | Val P@10: 0.00018
Epoch 08 | Loss: 11.5857 | Val P@10: 0.00028
Epoch 09 | Loss: 11.1184 | Val P@10: 0.00036
Epoch 10 | Loss: 10.6696 | Val P@10: 0.00041
Epoch 11 | Loss: 10.3482 | Val P@10: 0.00043
Epoch 12 | Loss: 10.0699 | Val P@10: 0.00043
Epoch 13 | Loss: 9.8622 | Val P@10: 0.00046
Epoch 14 | Loss: 9.6406 | Val P@10: 0.00043
Epoch 15 | Loss: 9.4856 | Val P@10: 0.00046


# Submission Generation


In [ ]:
model.load_state_dict(torch.load('best_sasrec.pth', map_location=device))
model.eval()

recommendations = []
# Need to use the raw original user IDs so we map correctly
all_original_users = df['user_id'].unique()

with torch.no_grad():
    for user_id in all_original_users:
        user_idx = user2idx[user_id]
        
        # Use entire known history (train + val) to predict the future
        seq = full_user_history.get(user_id, []) 
        
        if len(seq) > max_seq_len:
            seq = seq[-max_seq_len:]
        else:
            seq = [0] * (max_seq_len - len(seq)) + seq
            
        seq_tensor = torch.tensor([seq], device=device)
        logits = model(seq_tensor).squeeze()
        
        # Mask out padding and ALL past read items
        logits[0] = -float('inf')
        read_items = full_user_history.get(user_id, [])
        if read_items:
            logits[read_items] = -float('inf')
            
        top_10_idx = torch.topk(logits, k=10).indices.cpu().numpy()
        
        # REMEMBER: We shifted indices by +1, so we must shift back by -1 to get real items
        top_10_item_ids = [idx2item[idx - 1] for idx in top_10_idx]
        
        recommendations.append({
            'user_id': user_id,
            'recommendation': " ".join([str(i) for i in top_10_item_ids]) 
        })

submission_df = pd.DataFrame(recommendations)
submission_df['user_id'] = pd.to_numeric(submission_df['user_id'])
submission_df = submission_df.sort_values(by='user_id').reset_index(drop=True)
submission_df.to_csv(Path.cwd().parent/"submissions"/'submission_SASRec.csv', index=False)

# With data augmentation
## Imports

In [10]:
import os
import pandas as pd
import numpy as np
import torch
import requests
import time
import re
from tqdm import tqdm
import pickle
from sentence_transformers import SentenceTransformer

## Data augmentation (OenLibrary API)

In [12]:
try:
    items_df = pd.read_csv(Path.cwd().parent/"data"/"items_augmented.csv")
    print(f"Augmented file found, skipping API calls.")
except Exception as e:
    print("Augmented data not found. Fetching from OpenLibrary API")
    augmented_file = Path.cwd().parent/"data"/"items_augmented.csv"
    items_df = pd.read_csv(Path.cwd().parent/"data"/"items.csv")
    items_df = items_df.rename(columns={"i" : "item_id"})
    descriptions, page_counts, years = [], [], []

    # Use a generic email for the User-Agent as requested by OpenLibrary
    headers = {'User-Agent': 'UniversityLibraryRecommender/1.0 (mailto:julien.delez@epfl.ch)'}

    for isbn in tqdm(items_df['ISBN Valid']):
        desc, pages, year = "", 0, ""
        if pd.notna(isbn) and str(isbn).strip() != "":
            clean_isbn = str(isbn).replace('-', '').strip()
            url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{clean_isbn}&format=json&jscmd=data"
            
            try:
                response = requests.get(url, headers=headers, timeout=5)
                if response.status_code == 200:
                    data = response.json()
                    book_key = f"ISBN:{clean_isbn}"
                    if book_key in data:
                        book_data = data[book_key]
                        pages = book_data.get('number_of_pages', 0)
                        
                        pub_date = book_data.get('publish_date', '')
                        if pub_date:
                            match = re.search(r'\d{4}', pub_date)
                            if match: year = match.group(0)
                                
                        desc = book_data.get('notes', '')
                        if not desc:
                            url_record = book_data.get('url', '')
                            desc = f"See full record at: {url_record}" if url_record else ""
            except requests.exceptions.RequestException:
                pass # Silently handle connection errors to keep the loop going
                
        descriptions.append(str(desc))
        page_counts.append(pages)
        years.append(year)
        
        time.sleep(0.1) # Be polite to the API
        
    items_df['description'] = descriptions
    items_df['page_count'] = page_counts
    items_df['published_year'] = years
    items_df.to_csv(augmented_file, index=False)
    print(f"Saved augmented data to {augmented_file}")

Augmented data not found. Fetching from OpenLibrary API


  0%|          | 12/15291 [00:53<19:48:31,  4.67s/it]

## Generate text embeddings

In [ ]:
try:
    with open(Path.cwd().parent/"data"/"item_text_embeddings.pkl", 'rb') as f:
        item_id_to_embedding = pickle.load(f)
    print(f"Embedding file found, loading pre-computed embeddings.")
except Exception as e:
    print("Embeddings not found. Generating dense semantic vectors...")
    
    # Fill missing values to prevent concatenation errors
    cols_to_fill = ['title', 'authors', 'topic', 'description']
    items_df[cols_to_fill] = items_df[cols_to_fill].fillna("")
    
    items_df['combined_text'] = (
        "Title: " + items_df['title'] + ". " +
        "Author: " + items_df['authors'] + ". " +
        "Topic: " + items_df['topic'] + ". " +
        "Description: " + items_df['description']
    )
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device : {device}")
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    text_list = items_df['combined_text'].tolist()
    embeddings = model.encode(text_list, batch_size=128, show_progress_bar=True)
    
    item_id_to_embedding = {}
    for idx, item_id in enumerate(items_df['item_id']):
        item_id_to_embedding[item_id] = embeddings[idx]
        
    embeddings_file = Path.cwd().parent/"data"/"item_text_embeddings.pkl"
    with open(embeddings_file, 'wb') as f:
        pickle.dump(item_id_to_embedding, f)
    print(f"Saved text embeddings to {embeddings_file}")

# Semantic content-based filtering

In [ ]:
df = pd.read_csv(Path.cwd().parent/"data"/"interactions_train.csv")
df = df.rename(columns={"u": "user_id", "i": "item_id", "t": "timestamp"})
items_df = items_df.rename(columns={"i" : "item_id"})
# Create integer mappings
user2idx = {u: i for i, u in enumerate(df['user_id'].unique())}
idx2user = {i: u for i, u in enumerate(df['user_id'].unique())}
item2idx = {i: idx for i, idx in enumerate(items_df['item_id'].unique())}
idx2item = {idx: i for i, idx in enumerate(items_df['item_id'].unique())}

num_users = len(user2idx)
num_items = len(item2idx)

df['user_idx'] = df['user_id'].map(user2idx)
df['item_idx'] = df['item_id'].map(item2idx)

# Group interactions to get user history
user_history = df.dropna(subset=['item_idx']).groupby('user_idx')['item_idx'].apply(list).to_dict()
user_history_set = df.dropna(subset=['item_idx']).groupby('user_idx')['item_idx'].apply(set).to_dict()

# Build the Item Embedding Matrix in PyTorch
embed_dim = 384
item_embedding_matrix = torch.zeros((num_items, embed_dim))

for item_id, emb in item_id_to_embedding.items():
    if item_id in item2idx:
        idx = item2idx[item_id]
        item_embedding_matrix[idx] = torch.tensor(emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
item_embedding_matrix = item_embedding_matrix.to(device)

# Normalize item embeddings for Cosine Similarity
item_embedding_matrix = torch.nn.functional.normalize(item_embedding_matrix, p=2, dim=1)

# Generate Recommendations
recommendations = []
print(f"Generating Semantic Recommendations for {num_users} users on {device}")

for user_idx in tqdm(range(num_users)):
    history_indices = user_history.get(user_idx, [])
    
    if len(history_indices) == 0:
        # Fallback for users with no mapped history
        top_10_idx = np.random.choice(num_items, 10, replace=False)
    else:
        # 1. Get embeddings for read books
        history_embs = item_embedding_matrix[history_indices]
        
        # 2. Average to create User Profile and normalize
        user_profile = torch.mean(history_embs, dim=0, keepdim=True)
        user_profile = torch.nn.functional.normalize(user_profile, p=2, dim=1)
        
        # 3. Compute Cosine Similarity scores
        scores = torch.matmul(user_profile, item_embedding_matrix.T).squeeze()
        
        # 4. Mask out already read books
        read_items = list(user_history_set.get(user_idx, set()))
        if read_items:
            scores[read_items] = -float('inf')
            
        # 5. Extract Top 10
        top_10_idx = torch.topk(scores, k=10).indices.cpu().numpy()
        
    original_user_id = idx2user[user_idx]
    top_10_item_ids = [idx2item[idx] for idx in top_10_idx]
    
    recommendations.append({
        'user_id': original_user_id,
        'top_10': " ".join([str(i) for i in top_10_item_ids]) 
    })

# Format and save for Kaggle
submission_df = pd.DataFrame(recommendations)
submission_df['user_id'] = pd.to_numeric(submission_df['user_id'])
submission_df = submission_df.sort_values(by='user_id').reset_index(drop=True)

output_file = 'submission_semantic_content.csv'
submission_df.to_csv(output_file, index=False)
print(f"Success! Saved final predictions to {output_file}.")